<a href="https://colab.research.google.com/github/toche7/AI_ITM/blob/main/LabFinetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1: ติดตั้ง Unsloth (ใช้เวลา ~3-5 นาที)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" trl peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-ic5clsig/unsloth_6199715c4f2e4d98b0066e13b05166f0
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-ic5clsig/unsloth_6199715c4f2e4d98b0066e13b05166f0
  Resolved https://github.com/unslothai/unsloth.git to commit e775f941a42ec6daf06ed95ad94d5adc7bb37247
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached xformers-0.0.26.post1.tar.gz (4.1 MB)
  Preparing metadata (setup.py) ... done
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for xformers
  Running setup.py clean for xformers
Failed to build xformers
ERROR: ERROR: Failed to build installable w

In [2]:
# Cell 2: ตรวจสอบ GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
# ควรเห็น: GPU: Tesla T4 | VRAM: 15.8 GB

GPU: Tesla T4
VRAM: 15.6 GB


In [3]:
# Step 2
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = "unsloth/Meta-Llama-3.1-8B-Instruct",
    max_seq_length = 2048,    # ความยาว context สูงสุด
    dtype         = None,     # auto-detect (bfloat16 บน T4)
    load_in_4bit  = True,     # QLoRA: quantize เป็น 4-bit NF4
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/meta-llama-3.1-8b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


In [4]:
# Step 3
model = FastLanguageModel.get_peft_model(
    model,
    r                        = 8,     # Reduced from 16 to save VRAM
    lora_alpha               = 16,    # Adjusted (2x r)
    lora_dropout             = 0.05,
    target_modules           = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    bias                     = "none",
    use_gradient_checkpointing = "unsloth",
    random_state             = 3407,
)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.5.2 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


In [5]:
# Step 4
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

# โหลด dataset (alpaca format: instruction / input / output)
dataset  = load_dataset("yahma/alpaca-cleaned", split="train[:500]")
tokenizer = get_chat_template(tokenizer, chat_template="llama-3")

def format_alpaca(examples):
    """แปลง Alpaca format → chat messages → formatted text"""
    texts = []
    for inst, inp, out in zip(examples["instruction"],
                              examples["input"],
                              examples["output"]):
        user_msg = inst if not inp else f"{inst}\n\n{inp}"
        convo = [
            {"role": "system",    "content": "You are a helpful assistant."},
            {"role": "user",      "content": user_msg},
            {"role": "assistant", "content": out},
        ]
        texts.append(tokenizer.apply_chat_template(
            convo, tokenize=False, add_generation_prompt=False))
    return {"text": texts}

dataset    = dataset.map(format_alpaca, batched=True)
split      = dataset.train_test_split(test_size=0.1, seed=42)
train_data = split["train"]   # 450 ตัวอย่าง
eval_data  = split["test"]    # 50 ตัวอย่าง
print(dataset[0]["text"][:300])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>

Give three tips for staying healthy.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

1. Eat a balanced and nutritious diet: Make sure your meals are in


In [12]:
# Step 5
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model           = model,
    tokenizer       = tokenizer,
    train_dataset   = train_data,
    eval_dataset    = eval_data,
    dataset_text_field = "text",
    max_seq_length  = 1024, # Reduced to 1024 to save VRAM on T4
    args = TrainingArguments(
        per_device_train_batch_size  = 1, # Reduced to 1 to avoid OOM
        gradient_accumulation_steps  = 8, # 1x8=8 effective batch
        num_train_epochs             = 3,
        learning_rate                = 2e-4,
        lr_scheduler_type            = "cosine",
        warmup_ratio                 = 0.05,
        fp16                         = not torch.cuda.is_bf16_supported(),
        bf16                         = torch.cuda.is_bf16_supported(),
        eval_strategy                = "steps",
        eval_steps                   = 50,
        logging_steps                = 10,
        output_dir                   = "outputs",
        save_strategy                = "steps", # Match eval_strategy
        save_steps                   = 50,    # Save every 50 steps to pick best
        load_best_model_at_end       = True,
        metric_for_best_model       = "eval_loss",
    ),
)
trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 450 | Num Epochs = 3 | Total steps = 171
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 20,971,520 of 8,051,232,768 (0.26% trained)


Step,Training Loss,Validation Loss
50,0.695588,1.072005
100,0.300476,1.335857
150,0.168467,1.560053
171,0.162105,1.562794


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-50/tokenizer_config.json.
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionM

TrainOutput(global_step=171, training_loss=0.34089281131126725, metrics={'train_runtime': 1160.6034, 'train_samples_per_second': 1.163, 'train_steps_per_second': 0.147, 'total_flos': 9947413498945536.0, 'train_loss': 0.34089281131126725, 'epoch': 3.0})

In [13]:
FastLanguageModel.for_inference(model)  # เร็วขึ้น 2×
messages = [{
    "role": "user",
    "content": "อธิบาย LoRA ให้คนที่ไม่รู้เรื่อง ML ฟังได้เข้าใจ"
}]
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")
outputs = model.generate(
    input_ids      = inputs,
    max_new_tokens = 512,
    temperature    = 0.7,      # ความ creative (0=deterministic, 1=สุ่ม)
    top_p          = 0.9,      # Nucleus sampling
    repetition_penalty = 1.1,  # ลดการพูดซ้ำ
)
print(tokenizer.decode(outputs[0][len(inputs[0]):]))

Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


LoRA (Low-Rank Adaptation) เป็นเทคนิคในการปรับเปลี่ยนโมเดล Machine Learning เพื่อให้เหมาะสมกับปัญหาใหม่ โดยใช้การลดขนาดของพารามิเตอร์โมเดลให้อยู่ในรูปแบบที่ต่ำลง ซึ่งจะทำให้สามารถปรับโมเดลให้เหมาะสมกับปัญหาใหม่ได้อย่างรวดเร็ว 

โดยทั่วไปแล้ว การปรับโมเดล Machine Learning จะต้องใช้เวลานานและข้อมูลจำนวนมาก แต่การใช้ LoRA จะช่วยให้การปรับโมเดลมีความเร็วขึ้น และสามารถปรับโมเดลได้กับปัญหาต่างๆ ที่แตกต่างกันได้อย่างรวดเร็ว 

อย่างไรก็ตาม ควรทราบว่าการใช้ LoRA นั้นมีข้อจำกัด คือ โมเดลหลังจากการปรับจะไม่ดีกว่าโมเดลที่ไม่มีการปรับ เนื่องจากการลดขนาดของพารามิเตอร์อาจทำให้ความแม่นยำของโมเดลดับต่ำลง ดังนั้นจึงควรเลือกใช้ LoRA อย่างระมัดระวงและเหมาะสมกับปัญหาใดๆ ที่ต้องการแก้ไข.<|eot_id|>
